This notebook to test the results of a trained segmentation model. This notebook will apply the segmentation model to any GLIMS_ID and produce a list of segmentations.

Next steps for 8/21:
3. Put everything back in the loop

#note this script assumes the files are in order in the director

In [53]:
import os
from datetime import datetime

import imageio.v2 as imageio
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import torch
import torch.nn as nn
import torchvision
from helpers import preprocess, read
from inference.cnn import UNet, conv_block
from PIL import Image
from rasterio.mask import mask
from skimage.filters import gaussian
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm

device = 'cuda' if torch.cuda.is_available() else 'cpu'
device = torch.device(device)

In [54]:
torch_model = torch.load("unet_summer_model_unfrozen_100", map_location=torch.device('cpu')) #for cpu
torch_model.to(device)
torch_model.eval()

UNet(
  (resnet): Sequential(
    (0): Conv2d(9, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (4): Sequential(
      (0): Bottleneck(
        (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (downsample): Sequential(
          (0): Conv2d(64, 256,

In [55]:
#define inputs

common_bands = ['blue','green','red','nir','swir','thermal'] # we will later filter this down further, if needed
dim = (128,128)

data_label = "full_time_series_c02_t1_l2"
glacier_view_dir = os.path.join(os.path.expanduser("~"),
    "Desktop", "projects", "GlacierView")
ee_data_dir = os.path.join(glacier_view_dir,  "src", "earth_engine","data","ee_landing_zone",data_label)
landsat_dir = '/Volumes/T7/GlacierView/ee_landing_zone/full_time_series_c02_t1_l2/landsat' # External SSD
# landsat_dir = '/Users/mattw/Desktop/projects/GlacierView/src/earth_engine/data/ee_landing_zone/full_time_series_c02_t1_l2/landsat' # Internal SSD
dem_dir = os.path.join(ee_data_dir, "dems")
masks_dir = os.path.join(glacier_view_dir, "src","segmentation","training","data","masks_staging_2")


In [ ]:
metadata_dir = os.path.join(glacier_view_dir,"src","earth_engine","data","processed_metadata",data_label)
filtered_inference_df = pd.read_csv(os.path.join(metadata_dir,"filtered_inference_data.csv"))
print(filtered_inference_df.shape)

## make sure we only run inference for glaciers we have data for
filtered_inference_df = filtered_inference_df[filtered_inference_df.glims_id.isin(os.listdir(landsat_dir))]

filtered_inference_df[filtered_inference_df.glims_id == 'G007026E45991N'] ## FORCE TO ONE GLACIER

# this is the loop we'd use if we wanted to run inference for all glaciers
for glims_id in np.unique(filtered_inference_df.glims_id):
    print(glims_id)

    ##### PREPROCESSING #####

    PROB_THRESH = 0.5
    SMOOTH_FACTOR = 0

    #prepare paths
    glacier_dir = os.path.join(landsat_dir, glims_id)
    dem_path = os.path.join(dem_dir, f"{glims_id}_NASADEM.tif")
    mask_file_name = f"{glims_id}.tif"

    #read mask
    mask = Image.open(os.path.join(masks_dir, mask_file_name))
    mask = np.expand_dims(np.array(mask),2)
    mask = {mask_file_name: mask}

    #preprocess masks
    mask = preprocess.resize_rasters(mask,dim)
    mask = next(iter(mask.values()))

    #read images
    images = read.get_rasters(glacier_dir,filtered_file_path=os.path.join(metadata_dir,"filtered_inference_data.csv"))
    # single_key = list(images.keys())[0]
    original_sizes = [images[key].shape[:-1] for key in images]

    #preprocess images
    nimages = preprocess.get_common_bands(images,common_bands)
    nimages = preprocess.normalize_rasters(nimages)
    nimages = preprocess.resize_rasters(nimages,dim)

    #read dems
    dem = read.get_dem(dem_path)
    dem_file_name = next(iter(dem))

    #preprocess dems
    dem = preprocess.resize_rasters(dem, dim)
    dem = preprocess.normalize_rasters(dem)

    ######################################################################################################################################

    ##### Combine images and dems #####
    combined_images_and_dems = [np.concatenate((nimages[file_name], dem[dem_file_name]),axis = 2) for file_name in nimages]

    ##### Further process data #####
    X = np.stack(combined_images_and_dems)
    X = np.nan_to_num(X, copy=True, nan=0.0)

    ##### Get dates for time series #####
    image_file_names_ordered = sorted(nimages.keys())
    image_dates = [datetime.strptime(f.split("_")[1],'%Y-%m-%d') for f in image_file_names_ordered]

    ##### #####
    inputs = torch.tensor(X)
    inputs = inputs.permute(0,3,1,2) # n_images, bands, height, width
    inputs.shape

    ##### Isolate bands for ndsi and ndwi calculations #####
    green = inputs[:,1,:,:]
    nir = inputs[:,3,:,:]
    swir = inputs[:,4,:,:]

    ##### Add NDSI to inputs tensor #####
    ndsi = (green - swir)/(green + swir+SMOOTH_FACTOR)
    ndsi = torch.tensor(np.nan_to_num(ndsi, copy=True, nan=0.0))
    ndsi = ndsi.unsqueeze(dim=1) # takes [n_images, height, width] to [n_images, bands, height, width]
    inputs = torch.cat((ndsi, inputs), dim=1)

    ##### Add NDWI to inputs tensor #####
    ndwi = (green - nir)/(green + nir+SMOOTH_FACTOR)
    ndwi = torch.tensor(np.nan_to_num(ndwi, copy=True, nan=0.0))
    ndwi = ndwi.unsqueeze(dim=1) # takes [n_images, height, width] to [n_images, bands, height, width]
    inputs = torch.cat((ndwi, inputs), dim=1)

    ##### Optional smoothing
    ten_percent_smooth_param = round(inputs.shape[0]*.10)
    inputs = torch.tensor(gaussian(inputs, sigma = [ten_percent_smooth_param,0,0,0], mode = 'nearest'))

    ##### Filter image to adhere to the model bands #####
    inputs_filtered_bands = inputs[:,:,:,:]

    ##### create your dataloader #####
    prediction_dataset = TensorDataset(inputs_filtered_bands) # create your datset
    prediction_dataloader = DataLoader(prediction_dataset, batch_size=64,shuffle=False) # create your dataloader

    ##### Obtain predictions #####
    predictions = []
    for i in tqdm(prediction_dataloader):
        with torch.no_grad():
            outputs = torch_model.forward(i[0].to(device=device,dtype=torch.float))
            outputs = torch.softmax(outputs, dim=1)
            outputs = outputs[:,1,:,:].unsqueeze(1)
        m = torch.nn.Threshold(PROB_THRESH, 0)
        predictions.append(m(outputs))
    predictions = torch.cat(predictions, dim=0)

    ##### Resize to original image size and obtain area estimates #####
    resized_predictions = []
    area_per_pred = []
    resized_imgs = []
    for index, size in enumerate(original_sizes):
        resize = torchvision.transforms.Resize(size, antialias=True)
        pred = resize(predictions[index])
        resized_imgs.append(pred.detach().cpu().numpy()[0])
        area = pred.sum().detach().cpu().numpy()
        area = area*0.0009 #to convert area
        area_per_pred.append(area)
        resized_predictions.append(resize(predictions[index]))


    ##### Create time series #####
    df = pd.DataFrame(area_per_pred, image_dates)
    df[glims_id] = df[0]
    df.drop([0],axis=1)
    df =  df[glims_id]
    df = df.to_frame()
    df.index.names = ['Dates']
    df.to_csv(f"{glims_id}_areas.csv")

    ts_output_dir = "time_series"
    is_exist = os.path.exists(ts_output_dir)
    if not is_exist:
       os.makedirs(ts_output_dir)
    df.plot(figsize=(10,5), ylabel="Area in km sq.", xlabel="Years", title=f"Area vs Time plot for {glims_id} Glacier")
    mo = df.groupby(pd.PeriodIndex(df.index, freq="Y"))[df.columns[0]].mean()
    mo.plot()
    plt.legend(["Area at each time point","Annual Mean Area"])
    plt.savefig(os.path.join(ts_output_dir, f"{glims_id}.png"))

    ##### Create GIF #####
    predictions = predictions.detach().cpu().numpy()

    gif_creation_dir = os.path.join("tmp", "gif_creation")
    is_exist = os.path.exists(gif_creation_dir)
    if not is_exist:
       os.makedirs(gif_creation_dir)

    gif_output_dir = "gifs"
    is_exist = os.path.exists(gif_output_dir)
    if not is_exist:
       os.makedirs(gif_output_dir)

    # gif_creation_dir = os.path.join(os.path.expanduser("~"), "PycharmProjects","glacier-view-analysis", "src", "segmentation","tmp","gif_creation")
    # gif_output_dir = os.path.join(os.path.expanduser("~"), "PycharmProjects","glacier-view-analysis", "src", "segmentation", "gifs")

    for f in os.listdir(gif_creation_dir):
        os.remove(os.path.join(gif_creation_dir, f))

    for i in range(inputs.shape[0]):
        fig, axs = plt.subplots(2,3, figsize=(10,10)) ##update the number of suplots to equal the number of layers you want to display
        fig.suptitle(image_file_names_ordered[i])
        axs[0,0].imshow(inputs[i,:,:,:][[4,3,2],:,:].permute(1,2,0))
        axs[0,0].title.set_text('RGB')
        axs[0,1].imshow(inputs[i,:,:,:][1,:,:])
        axs[0,1].title.set_text('NDSI')
        axs[0,2].imshow(inputs[i,:,:,:][7,:,:])
        axs[0,2].title.set_text('Thermal')
        axs[1,0].imshow(inputs[i,:,:,:][8,:,:])
        axs[1,0].title.set_text('DEM')
        axs[1,1].imshow(inputs[i,:,:,:][[4,3,2],:,:].permute(1,2,0))
        axs[1,1].imshow(predictions[i,0,:,:],alpha=0.5)
        axs[1,1].title.set_text('Prediction')
        axs[1,2].imshow(mask)
        axs[1,2].title.set_text('Ground truth')

        plt.savefig(os.path.join(gif_creation_dir,f'{image_file_names_ordered[i]}_final.png'), dpi = 100)
        # plt.show()

    ##### SAVE GIF #####
    with imageio.get_writer(os.path.join(gif_output_dir,f"{glims_id}_final_NEWEST.gif"), mode='I') as writer:
        for filename in sorted(os.listdir(gif_creation_dir)):
            image = imageio.imread(os.path.join(gif_creation_dir,filename))
            writer.append_data(image)

(16406, 20)
G007026E45991N


/Users/mattw/.venv/glaciers-pytorch/lib/python3.9/site-packages/torchvision/transforms/functional.py:1603: UserWarning: The default value of the antialias parameter of all the resizing transforms (Resize(), RandomResizedCrop(), etc.) will change from None to True in v0.17, in order to be consistent across the PIL and Tensor backends. To suppress this warning, directly pass antialias=True (recommended, future default), antialias=None (current default, which means False for Tensors and True for PIL), or antialias=False (only works on Tensors - PIL will still use antialiasing). This also applies if you are using the inference transforms from the models weights: update the call to weights.transforms(antialias=True).
  warnings.warn(
/Users/mattw/.venv/glaciers-pytorch/lib/python3.9/site-packages/torchvision/transforms/functional.py:1603: UserWarning: The default value of the antialias parameter of all the resizing transforms (Resize(), RandomResizedCrop(), etc.) will change from None to Tr

G007050E45972N


/Users/mattw/.venv/glaciers-pytorch/lib/python3.9/site-packages/torchvision/transforms/functional.py:1603: UserWarning: The default value of the antialias parameter of all the resizing transforms (Resize(), RandomResizedCrop(), etc.) will change from None to True in v0.17, in order to be consistent across the PIL and Tensor backends. To suppress this warning, directly pass antialias=True (recommended, future default), antialias=None (current default, which means False for Tensors and True for PIL), or antialias=False (only works on Tensors - PIL will still use antialiasing). This also applies if you are using the inference transforms from the models weights: update the call to weights.transforms(antialias=True).
  warnings.warn(
/Users/mattw/.venv/glaciers-pytorch/lib/python3.9/site-packages/torchvision/transforms/functional.py:1603: UserWarning: The default value of the antialias parameter of all the resizing transforms (Resize(), RandomResizedCrop(), etc.) will change from None to Tr

G007281E45526N


...

/Users/mattw/.venv/glaciers-pytorch/lib/python3.9/site-packages/torchvision/transforms/functional.py:1603: UserWarning: The default value of the antialias parameter of all the resizing transforms (Resize(), RandomResizedCrop(), etc.) will change from None to True in v0.17, in order to be consistent across the PIL and Tensor backends. To suppress this warning, directly pass antialias=True (recommended, future default), antialias=None (current default, which means False for Tensors and True for PIL), or antialias=False (only works on Tensors - PIL will still use antialiasing). This also applies if you are using the inference transforms from the models weights: update the call to weights.transforms(antialias=True).
  warnings.warn(
/Users/mattw/.venv/glaciers-pytorch/lib/python3.9/site-packages/torchvision/transforms/functional.py:1603: UserWarning: The default value of the antialias parameter of all the resizing transforms (Resize(), RandomResizedCrop(), etc.) will change from None to Tr

G087182E36381N


/Users/mattw/.venv/glaciers-pytorch/lib/python3.9/site-packages/torchvision/transforms/functional.py:1603: UserWarning: The default value of the antialias parameter of all the resizing transforms (Resize(), RandomResizedCrop(), etc.) will change from None to True in v0.17, in order to be consistent across the PIL and Tensor backends. To suppress this warning, directly pass antialias=True (recommended, future default), antialias=None (current default, which means False for Tensors and True for PIL), or antialias=False (only works on Tensors - PIL will still use antialiasing). This also applies if you are using the inference transforms from the models weights: update the call to weights.transforms(antialias=True).
  warnings.warn(
/Users/mattw/.venv/glaciers-pytorch/lib/python3.9/site-packages/torchvision/transforms/functional.py:1603: UserWarning: The default value of the antialias parameter of all the resizing transforms (Resize(), RandomResizedCrop(), etc.) will change from None to Tr

G087347E36344N


/Users/mattw/.venv/glaciers-pytorch/lib/python3.9/site-packages/torchvision/transforms/functional.py:1603: UserWarning: The default value of the antialias parameter of all the resizing transforms (Resize(), RandomResizedCrop(), etc.) will change from None to True in v0.17, in order to be consistent across the PIL and Tensor backends. To suppress this warning, directly pass antialias=True (recommended, future default), antialias=None (current default, which means False for Tensors and True for PIL), or antialias=False (only works on Tensors - PIL will still use antialiasing). This also applies if you are using the inference transforms from the models weights: update the call to weights.transforms(antialias=True).
  warnings.warn(
/Users/mattw/.venv/glaciers-pytorch/lib/python3.9/site-packages/torchvision/transforms/functional.py:1603: UserWarning: The default value of the antialias parameter of all the resizing transforms (Resize(), RandomResizedCrop(), etc.) will change from None to Tr